# LeetCode 1762 — Buildings With an Ocean View bằng Python

Link đề: https://leetcode.com/problems/buildings-with-an-ocean-view/ · Độ khó: **Medium** · Chủ đề: Array, Stack, Monotonic Stack

> **Ghi chú (bài Premium 🔒):** đề trong notebook này được viết lại theo hiểu biết về bài và theo bản mô tả công khai của kho doocs/leetcode. Nếu bạn có tài khoản LeetCode Premium, hãy đối chiếu lại câu chữ, ví dụ và ràng buộc với đề gốc.

Skeleton LeetCode:

```python
class Solution:
    def findBuildings(self, heights: List[int]) -> List[int]:
        ...
```

## 1. Mục tiêu học

1. Dịch điều kiện "nhìn thấy biển" thành điều kiện toán học: **mọi** tòa nhà bên phải **thấp hơn hẳn**.
2. Từ vét cạn `O(n²)` rút ra ý tưởng **max hậu tố** (suffix maximum) và **duyệt từ phải sang trái**.
3. Cài lời giải bằng **stack đơn điệu** khi buộc phải duyệt từ trái sang phải (dạng "online").
4. Chú ý chi tiết **nghiêm ngặt** (`>` chứ không phải `≥`) và yêu cầu trả chỉ số **tăng dần**.
5. Kiểm thử đối chiếu nhiều cách trên mảng ngẫu nhiên có nhiều giá trị trùng.

## 2. Đề bài & ví dụ

Có `n` tòa nhà xếp thành một hàng, `heights[i]` là chiều cao tòa nhà `i`. **Biển nằm ở bên phải** các tòa nhà. Tòa nhà `i` **nhìn thấy biển** nếu mọi tòa nhà ở bên phải nó đều có chiều cao **nhỏ hơn hẳn** `heights[i]`.

Trả về danh sách **chỉ số** (0-based) các tòa nhà nhìn thấy biển, **sắp tăng dần**.

**Ví dụ 1:** `heights = [4,2,3,1]` → `[0,2,3]`. Tòa 1 (cao 2) bị tòa 2 (cao 3) che.

**Ví dụ 2:** `heights = [4,3,2,1]` → `[0,1,2,3]`.

**Ví dụ 3:** `heights = [1,3,2,4]` → `[3]`.

**Ràng buộc:** `1 ≤ heights.length ≤ 10⁵`, `1 ≤ heights[i] ≤ 10⁹`.

In [1]:
import random
import time
from typing import List

examples = [([4, 2, 3, 1], [0, 2, 3]), ([4, 3, 2, 1], [0, 1, 2, 3]), ([1, 3, 2, 4], [3])]
for h, exp in examples:
    print(f"heights={h} -> kỳ vọng {exp}")

heights=[4, 2, 3, 1] -> kỳ vọng [0, 2, 3]
heights=[4, 3, 2, 1] -> kỳ vọng [0, 1, 2, 3]
heights=[1, 3, 2, 4] -> kỳ vọng [3]


## 3. Kiến thức nền

### 3.1 `all(...)` trên một lát cắt

`all(heights[j] < heights[i] for j in range(i + 1, n))` kiểm tra "mọi tòa bên phải thấp hơn". Với `i = n − 1` dãy rỗng ⇒ `all` trả `True` (tòa cuối luôn thấy biển).

In [2]:
h = [4, 2, 3, 1]
for i in range(len(h)):
    print(i, h[i], all(h[j] < h[i] for j in range(i + 1, len(h))))
print("all([]) =", all([]))

0 4 True
1 2 False
2 3 True
3 1 True
all([]) = True


### 3.2 Duyệt ngược và `range(n − 1, −1, −1)`

`range(n - 1, -1, -1)` đi `n−1, n−2, …, 0`. `reversed(range(n))` tương đương.

In [3]:
print(list(range(4 - 1, -1, -1)), list(reversed(range(4))))

[3, 2, 1, 0] [3, 2, 1, 0]


### 3.3 Đảo list: `lst[::-1]` và `lst.reverse()`

Thu thập chỉ số khi duyệt ngược sẽ ra thứ tự **giảm dần**; cần đảo lại trước khi trả. `lst.reverse()` đảo tại chỗ `O(n)`; `lst[::-1]` tạo bản sao.

In [4]:
a = [3, 2, 0]; b = a[::-1]; a.reverse()
print(a, b)

[0, 2, 3] [0, 2, 3]


### 3.4 Stack bằng `list`

`append` = push, `pop()` = lấy phần tử trên cùng, `st[-1]` = xem đỉnh. Stack **đơn điệu** giữ phần tử theo thứ tự tăng/giảm bằng cách `pop` những phần tử vi phạm trước khi `push`.

In [5]:
st = []
for x in [5, 3, 4, 1]:
    while st and st[-1] <= x:        # giữ stack giảm nghiêm ngặt
        st.pop()
    st.append(x)
    print("sau khi thêm", x, ":", st)

sau khi thêm 5 : [5]
sau khi thêm 3 : [5, 3]
sau khi thêm 4 : [5, 4]
sau khi thêm 1 : [5, 4, 1]


## 4. Ý tưởng nền tảng / trực giác

Tòa `i` thấy biển ⇔ `heights[i] > max(heights[i+1:])` (với tòa cuối, max của dãy rỗng coi như `0`, mà chiều cao `≥ 1`).

Tính tay **Ví dụ 1** từ **phải sang trái**, giữ `mx` = cao nhất đã thấy ở bên phải:

| i | heights[i] | mx bên phải | thấy biển? | mx mới |
|---|---|---|---|---|
| 3 | 1 | 0 | có | 1 |
| 2 | 3 | 1 | có | 3 |
| 1 | 2 | 3 | không | 3 |
| 0 | 4 | 3 | có | 4 |

Kết quả (đảo lại): `[0, 2, 3]`. Chỉ cần **một** con số `mx` thay vì xét lại cả đoạn bên phải.

In [6]:
h = [4, 2, 3, 1]
print([(i, h[i], max(h[i + 1:], default=0)) for i in range(len(h))])

[(0, 4, 3), (1, 2, 3), (2, 3, 1), (3, 1, 0)]


## 5. Cách 1 — Brute Force / baseline: kiểm tra từng tòa với mọi tòa bên phải

In [7]:
def ocean_bruteforce(heights):
    n = len(heights)
    return [i for i in range(n) if all(heights[j] < heights[i] for j in range(i + 1, n))]

for h, exp in examples:
    got = ocean_bruteforce(h); print(got, "kỳ vọng", exp); assert got == exp

[0, 2, 3] kỳ vọng [0, 2, 3]
[0, 1, 2, 3] kỳ vọng [0, 1, 2, 3]
[3] kỳ vọng [3]


**Độ phức tạp:** `O(n²)` tệ nhất (mảng giảm dần: mọi tòa đều phải quét hết bên phải). `n = 10⁵` ⇒ `5·10⁹` phép so sánh ⇒ **TLE**.

**Nút thắt:** câu hỏi "mọi tòa bên phải thấp hơn?" lặp lại cho mọi `i`, nhưng chỉ phụ thuộc vào **max** của đoạn bên phải — một đại lượng cập nhật được dần dần.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** `ok(i) ⇔ heights[i] > sufmax[i+1]` với `sufmax[k] = max(heights[k:])` (và `sufmax[n] = 0`). Và `sufmax[i] = max(heights[i], sufmax[i+1])` — tính từ phải sang trái trong `O(1)` mỗi bước.

**Quy tắc khi duyệt từ phải:**
- **Nếu** `heights[i] > mx` **thì** tòa `i` thấy biển; cập nhật `mx = heights[i]`.
- **Ngược lại** bỏ qua (`mx` không đổi).

**Phiên bản từ trái (stack đơn điệu):** duyệt `i` tăng dần, giữ stack các **ứng viên** (chỉ số chưa bị che). Khi gặp tòa `i`, mọi ứng viên có chiều cao `≤ heights[i]` bị **che vĩnh viễn** ⇒ `pop`. Sau đó `push i`. Cuối cùng stack (từ đáy lên đỉnh) chính là đáp án, đã tăng dần. Invariant: chiều cao trong stack **giảm nghiêm ngặt** từ đáy lên đỉnh.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mảng max hậu tố

In [8]:
def ocean_suffix(heights):
    n = len(heights)
    suf = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        suf[i] = max(heights[i], suf[i + 1])
    return [i for i in range(n) if heights[i] > suf[i + 1]]

for h, _ in examples:
    print(h, "suffix max:", [max(h[i:]) for i in range(len(h))], "->", ocean_suffix(h))

[4, 2, 3, 1] suffix max: [4, 3, 3, 1] -> [0, 2, 3]
[4, 3, 2, 1] suffix max: [4, 3, 2, 1] -> [0, 1, 2, 3]
[1, 3, 2, 4] suffix max: [4, 4, 4, 4] -> [3]


**Code nộp LeetCode (Cách 2):** đổi tên hàm thành `findBuildings` trong lớp `Solution`. **Độ phức tạp:** Time `O(n)`, Space `O(n)` cho mảng `suf`.

### 7.2 Cách 3 — Duyệt từ phải với một biến `mx` (tối ưu bộ nhớ)

In [9]:
def ocean_scan_right(heights):
    res, mx = [], 0
    for i in range(len(heights) - 1, -1, -1):
        if heights[i] > mx:
            res.append(i)
            mx = heights[i]
    res.reverse()
    return res

print([ocean_scan_right(h) for h, _ in examples])

[[0, 2, 3], [0, 1, 2, 3], [3]]


**Code nộp LeetCode (Cách 3):** xem mục 9. **Độ phức tạp:** Time `O(n)`, Space `O(1)` ngoài kết quả.

### 7.3 Cách 4 — Stack đơn điệu duyệt từ trái

In [10]:
class SolutionStack:
    def findBuildings(self, heights: List[int]) -> List[int]:
        st = []
        for i, h in enumerate(heights):
            while st and heights[st[-1]] <= h:     # bị tòa i che (không thấp hơn hẳn)
                st.pop()
            st.append(i)
        return st

print([SolutionStack().findBuildings(h) for h, _ in examples])

[[0, 2, 3], [0, 1, 2, 3], [3]]


**Độ phức tạp Cách 4:** mỗi chỉ số được `push` một lần và `pop` tối đa một lần ⇒ Time `O(n)` (khấu hao), Space `O(n)`.

Ưu điểm: không cần biết trước toàn mảng — nếu các tòa nhà được "đưa vào" lần lượt từ trái sang (dữ liệu dạng luồng), stack luôn chứa đúng đáp án cho phần đã đọc.

### 7.4 Cách 5 — Max hậu tố bằng `itertools.accumulate` (viết gọn)

`accumulate(reversed(heights), max)` cho dãy max chạy từ phải sang. Dịch đi một vị trí để có "max **bên phải** (không tính chính nó)". Đây là phiên bản hàm-lập-trình của Cách 2.

**Giải thích từng đoạn:**
1. `rev_max = list(accumulate(reversed(h), max))` — `rev_max[t] = max(h[n−1−t:])`.
2. Max bên phải tòa `i` (không tính `i`) là `rev_max[n−2−i]` nếu `i < n−1`, ngược lại `0`.
3. Lọc các `i` có `h[i] >` giá trị đó.

In [11]:
from itertools import accumulate

def ocean_accumulate(heights):
    n = len(heights)
    rev_max = list(accumulate(reversed(heights), max))
    right = [rev_max[n - 2 - i] if i < n - 1 else 0 for i in range(n)]
    return [i for i in range(n) if heights[i] > right[i]]

for h, exp in examples:
    print(h, "-> max bên phải:", [max(h[i + 1:], default=0) for i in range(len(h))], "->", ocean_accumulate(h))
    assert ocean_accumulate(h) == exp

[4, 2, 3, 1] -> max bên phải: [3, 3, 1, 0] -> [0, 2, 3]
[4, 3, 2, 1] -> max bên phải: [3, 2, 1, 0] -> [0, 1, 2, 3]
[1, 3, 2, 4] -> max bên phải: [4, 4, 4, 0] -> [3]


**Độ phức tạp Cách 5:** Time `O(n)`, Space `O(n)`. Gọn nhưng dễ lệch chỉ số — nên tự kiểm bằng `max(h[i+1:], default=0)` như trên.

## 8. TRACE từng bước

Trace Cách 3 và Cách 4 trên **Ví dụ 1** `[4,2,3,1]`:

In [12]:
def trace_right(heights):
    res, mx = [], 0
    for i in range(len(heights) - 1, -1, -1):
        ok = heights[i] > mx
        if ok:
            res.append(i)
        print(f"i={i}: h={heights[i]}, mx bên phải={mx} -> {'THẤY biển' if ok else 'bị che'} | res (ngược)={res}")
        mx = max(mx, heights[i])
    res.reverse(); print("Kết quả:", res); return res

def trace_stack(heights):
    st = []
    for i, h in enumerate(heights):
        popped = []
        while st and heights[st[-1]] <= h:
            popped.append(st.pop())
        st.append(i)
        print(f"i={i}: h={h}, pop {popped} (bị che) -> stack={st} (cao: {[heights[j] for j in st]})")
    print("Kết quả:", st); return st

assert trace_right([4, 2, 3, 1]) == [0, 2, 3]
print()
assert trace_stack([4, 2, 3, 1]) == [0, 2, 3]

i=3: h=1, mx bên phải=0 -> THẤY biển | res (ngược)=[3]
i=2: h=3, mx bên phải=1 -> THẤY biển | res (ngược)=[3, 2]
i=1: h=2, mx bên phải=3 -> bị che | res (ngược)=[3, 2]
i=0: h=4, mx bên phải=3 -> THẤY biển | res (ngược)=[3, 2, 0]
Kết quả: [0, 2, 3]

i=0: h=4, pop [] (bị che) -> stack=[0] (cao: [4])
i=1: h=2, pop [] (bị che) -> stack=[0, 1] (cao: [4, 2])
i=2: h=3, pop [1] (bị che) -> stack=[0, 2] (cao: [4, 3])
i=3: h=1, pop [] (bị che) -> stack=[0, 2, 3] (cao: [4, 3, 1])
Kết quả: [0, 2, 3]


Trace ví dụ tự dựng có **chiều cao bằng nhau** `[2, 2, 2]` (ví dụ tự dựng — đối chiếu bằng brute force; nên đối chiếu lại đề gốc): chỉ tòa cuối thấy biển vì "thấp hơn hẳn".

In [13]:
assert trace_stack([2, 2, 2]) == ocean_bruteforce([2, 2, 2]) == [2]

i=0: h=2, pop [] (bị che) -> stack=[0] (cao: [2])
i=1: h=2, pop [0] (bị che) -> stack=[1] (cao: [2])
i=2: h=2, pop [1] (bị che) -> stack=[2] (cao: [2])
Kết quả: [2]


## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 3** — duyệt từ phải, một biến `mx`.

In [14]:
class Solution:
    def findBuildings(self, heights: List[int]) -> List[int]:
        ans = []
        mx = 0
        for i in range(len(heights) - 1, -1, -1):
            if heights[i] > mx:
                ans.append(i)
                mx = heights[i]
        ans.reverse()
        return ans

for h, exp in examples:
    got = Solution().findBuildings(h); print(got, exp); assert got == exp

[0, 2, 3] [0, 2, 3]
[0, 1, 2, 3] [0, 1, 2, 3]
[3] [3]


## 10. Vì sao thuật toán đúng

**Cách 3.** Invariant: trước khi xét `i`, `mx = max(heights[i+1:])` (hoặc `0` nếu rỗng). Đúng khi bắt đầu (`i = n−1`). Tòa `i` thấy biển ⇔ mọi tòa bên phải `< heights[i]` ⇔ `max(heights[i+1:]) < heights[i]` ⇔ `heights[i] > mx`. Sau bước, `mx` mới `= max(heights[i], mx) = max(heights[i:])` — invariant giữ cho `i − 1`. Kết quả thu theo `i` giảm, đảo lại thành tăng. ∎

**Cách 4.** Invariant sau khi xử lý `0..i`: stack = tập chỉ số `j ≤ i` sao cho mọi `k ∈ (j, i]` có `heights[k] < heights[j]` (tòa `j` thấy biển **nếu** mảng dừng ở `i`), xếp tăng dần, chiều cao giảm nghiêm ngặt.
- Tòa `i` mới: mọi `j` trong stack với `heights[j] ≤ heights[i]` mất tư cách mãi mãi (đã có tòa `i` không thấp hơn ở bên phải) ⇒ `pop`; vì chiều cao trong stack giảm dần nên các phần tử bị loại nằm liền ở đỉnh — `while` dừng đúng chỗ.
- Các `j` còn lại có `heights[j] > heights[i]` ⇒ vẫn đủ tư cách. `i` tự nó đủ tư cách (bên phải rỗng) ⇒ `push`.

Khi `i = n−1`, stack là đáp án. ∎

## 11. Phân tích Time / Space Complexity

Gọi `n = len(heights) ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(n²)` tệ nhất | `O(1)` phụ |
| 2. Mảng max hậu tố | `O(n)` | `O(n)` |
| 3. Duyệt phải + `mx` | `O(n)` | `O(1)` phụ |
| 4. Stack đơn điệu | `O(n)` khấu hao | `O(n)` |

In [15]:
rng = random.Random(1762)
cases = {"giảm dần": list(range(10 ** 5, 0, -1)), "ngẫu nhiên": [rng.randint(1, 10 ** 9) for _ in range(10 ** 5)]}
for name, h in cases.items():
    row = []
    for fname, f in [("suffix", ocean_suffix), ("scan", Solution().findBuildings), ("stack", SolutionStack().findBuildings)]:
        t = time.perf_counter(); r = f(h); row.append(f"{fname}: {len(r)} tòa, {(time.perf_counter()-t)*1000:.0f}ms")
    print(f"{name:<10}", " | ".join(row))
h = list(range(3000, 0, -1)); t = time.perf_counter(); ocean_bruteforce(h)
print(f"brute force n=3000 giảm dần: {(time.perf_counter()-t)*1000:.0f} ms (n=10⁵ sẽ chậm hơn ~1100 lần)")

giảm dần   suffix: 100000 tòa, 11ms | scan: 100000 tòa, 5ms | stack: 100000 tòa, 5ms
ngẫu nhiên suffix: 15 tòa, 9ms | scan: 15 tòa, 2ms | stack: 15 tòa, 9ms


brute force n=3000 giảm dần: 140 ms (n=10⁵ sẽ chậm hơn ~1100 lần)


## 12. Bộ test

In [16]:
ALL = {"suffix": ocean_suffix, "accumulate": ocean_accumulate, "scan": ocean_scan_right, "stack": SolutionStack().findBuildings, "submit": Solution().findBuildings}
def run_all(h):
    res = {k: tuple(f(h)) for k, f in ALL.items()}
    assert len(set(res.values())) == 1, (h, res)
    return list(next(iter(res.values())))

for h, exp in examples:
    assert run_all(h) == exp
edge = [([5], [0]), ([1, 1], [1]), ([2, 2, 2], [2]), ([1, 2, 3, 4], [3]), ([3, 1, 3], [2]), ([5, 1, 5, 1], [2, 3]),
        ([10 ** 9, 1], [0, 1])]
for h, exp in edge:
    assert ocean_bruteforce(h) == exp and run_all(h) == exp, h
rng = random.Random(17620)
for _ in range(2000):
    h = [rng.randint(1, rng.choice([3, 10, 10 ** 9])) for _ in range(rng.randint(1, 30))]
    assert run_all(h) == ocean_bruteforce(h), h
print("Ví dụ, edge case (có chiều cao bằng nhau), 2000 test ngẫu nhiên: OK")

Ví dụ, edge case (có chiều cao bằng nhau), 2000 test ngẫu nhiên: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Dùng `≥` thay vì `>` (cho phép bằng nhau)

In [17]:
def wrong_ge(heights):
    res, mx = [], 0
    for i in range(len(heights) - 1, -1, -1):
        if heights[i] >= mx:                           # SAI
            res.append(i); mx = heights[i]
    return res[::-1]

print("sai :", wrong_ge([2, 2, 2]), "| đúng:", Solution().findBuildings([2, 2, 2]))

sai : [0, 1, 2] | đúng: [2]


**Vì sao sai:** đề yêu cầu tòa bên phải **nhỏ hơn hẳn**; tòa cao bằng vẫn che. **Sửa:** `>`.

### Lỗi 2 — Quên đảo kết quả

In [18]:
def wrong_no_reverse(heights):
    res, mx = [], 0
    for i in range(len(heights) - 1, -1, -1):
        if heights[i] > mx:
            res.append(i); mx = heights[i]
    return res                                          # SAI: đang giảm dần

h, exp = examples[0]
print("sai :", wrong_no_reverse(h), "| đúng:", exp)

sai : [3, 2, 0] | đúng: [0, 2, 3]


**Vì sao sai:** đề yêu cầu chỉ số **tăng dần**. **Sửa:** `res.reverse()` hoặc `res[::-1]`.

### Lỗi 3 — Duyệt từ trái và so với max bên **trái**

In [19]:
def wrong_left_max(heights):
    res, mx = [], 0
    for i, h in enumerate(heights):
        if h > mx:                                      # SAI: đây là "nhìn thấy về bên trái"
            res.append(i); mx = h
    return res

h, exp = examples[0]
print("sai :", wrong_left_max(h), "| đúng:", exp)

sai : [0] | đúng: [0, 2, 3]


**Vì sao sai:** biển ở **bên phải**, điều kiện phụ thuộc max **bên phải**. Duyệt trái sang với max tiền tố trả lời câu hỏi khác. **Sửa:** duyệt từ phải, hoặc stack đơn điệu (Cách 4).

### Lỗi 4 — Stack: `pop` với điều kiện `<` thay vì `<=`

In [20]:
def wrong_stack(heights):
    st = []
    for i, h in enumerate(heights):
        while st and heights[st[-1]] < h:              # SAI: tòa bằng chiều cao cũng che
            st.pop()
        st.append(i)
    return st

print("sai :", wrong_stack([3, 1, 3]), "| đúng:", Solution().findBuildings([3, 1, 3]))

sai : [0, 2] | đúng: [2]


**Vì sao sai:** tòa `0` (cao 3) bị tòa `2` (cao 3) che nhưng không bị `pop`. **Sửa:** `<=`.

### Lỗi 5 — Gọi `max(heights[i+1:])` trong vòng lặp (quá chậm + lỗi dãy rỗng)

In [21]:
def slow_max_slice(heights):
    res = []
    for i in range(len(heights)):
        right = heights[i + 1:]
        if not right or heights[i] > max(right):       # thiếu 'not right' -> ValueError ở tòa cuối
            res.append(i)
    return res

try:
    max([])
except ValueError as e:
    print("max([]) ->", type(e).__name__, ":", e)
h = list(range(4000, 0, -1))
for f in (slow_max_slice, Solution().findBuildings):
    t = time.perf_counter(); r = f(h); print(f"{getattr(f, '__name__', '?'):<16} {len(r)} tòa, {(time.perf_counter()-t)*1000:.0f} ms")

max([]) -> ValueError : max() iterable argument is empty
slow_max_slice   4000 tòa, 70 ms
findBuildings    4000 tòa, 0 ms


**Vì sao sai:** mỗi lần cắt lát + `max` tốn `O(n)` ⇒ tổng `O(n²)`; ngoài ra `max([])` ném `ValueError` ở tòa cuối nếu quên xử lý dãy rỗng. **Sửa:** giữ `mx` chạy từ phải (Cách 3) — hoặc tối thiểu dùng `max(..., default=0)`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Mọi phần tử bên phải nhỏ hơn" ⇒ **max hậu tố** / duyệt từ phải.
- "Phần tử tiếp theo lớn hơn", "bị che bởi" ⇒ **stack đơn điệu**.
- Kết quả thu khi duyệt ngược ⇒ nhớ **đảo lại**.
- Luôn kiểm tra chữ "strictly" ⇒ `>` hay `≥`.
- Mảng có giá trị trùng là nơi lỗi `>`/`≥` lộ ra — test với `[2,2,2]`.

## 15. Biến thể của bài

1. **Biển ở bên trái**: duyệt từ trái với max tiền tố.
2. **Biển ở cả hai phía** (thấy biển nếu thấy ít nhất một phía): hợp hai tập.
3. **Đếm số tòa mỗi tòa nhìn thấy về bên phải**: **1944 Number of Visible People in a Queue** (stack đơn điệu).
4. **739 Daily Temperatures / 496 Next Greater Element**: stack đơn điệu tìm phần tử lớn hơn kế tiếp.
5. **Trả về số tòa thấy biển sau mỗi lần thêm tòa vào bên phải** (Bài tập 5).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | Kiểm tra mọi tòa bên phải | `O(n²)` | `O(1)` | Kiểm chứng |
| 2. Max hậu tố | Mảng `suf` | `O(n)` | `O(n)` | Dễ hiểu, cần nhiều truy vấn max hậu tố |
| 3. Duyệt phải + `mx` | Một biến | `O(n)` | `O(1)` | **Bản nộp** |
| 4. Stack đơn điệu | Loại ứng viên bị che | `O(n)` | `O(n)` | Dữ liệu đến từ trái (online) |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** (a) `[1]`; (b) `[3,3,2,2,1]`; (c) `[1,2,1,2]`; (d) `[7,1,6,2,5,3,4]`.

**Bài 2 (tính tay):** Trace stack (Cách 4) cho `[2, 5, 3, 3, 1]`: ghi các phần tử bị `pop` ở mỗi bước.

**Bài 3 (điền TODO):** Hoàn thiện `suffix_max(h)` trả mảng `suf` độ dài `n+1` với `suf[n] = 0`.

**Bài 4 (tự viết lại):** Viết `both_sides(heights)` trả chỉ số tăng dần các tòa nhìn thấy biển ở **bên trái hoặc bên phải** (đều theo nghĩa "thấp hơn hẳn").

**Bài 5 (mini challenge):** Các tòa được thêm lần lượt vào **bên phải** hàng. Viết lớp `OceanView` với `add(h)` trả **số tòa** đang thấy biển sau khi thêm, `O(1)` khấu hao mỗi lần.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def suffix_max_todo(h):
    n = len(h)
    suf = [0] * (n + 1)
    # TODO: for i từ n-1 về 0: suf[i] = max(h[i], suf[i+1])
    return suf

In [23]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def both_sides(heights):
    # TODO: tập thấy bên phải (duyệt phải) ∪ tập thấy bên trái (duyệt trái), sắp tăng
    pass

class OceanView:
    def __init__(self):
        # TODO: stack đơn điệu như Cách 4
        pass
    def add(self, h):
        # TODO: pop các tòa <= h, push, trả len(stack)
        pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: duyệt từ phải. Mức 2: nhớ `>`. Mức 3: (a) `[0]`; (b) `[1, 3, 4]`; (c) `[3]`; (d) `[0, 2, 4, 6]`.

**Bài 2.** Mức 1: `i=0` push. Mức 2: `i=1` (5) pop 0. Mức 3: `i=2` (3) không pop; `i=3` (3) pop 2; `i=4` (1) không pop ⇒ `[1, 3, 4]`.

**Bài 3.** Mức 1: duyệt ngược. Mức 2: `max(h[i], suf[i+1])`. Mức 3: trả `suf`.

**Bài 4.** Mức 1: thấy bên phải như bài gốc. Mức 2: thấy bên trái: duyệt từ trái với `mx` tiền tố. Mức 3: `sorted(set(A) | set(B))`.

**Bài 5.** Mức 1: tòa mới luôn thấy biển. Mức 2: tòa cũ `≤ h` mất tầm nhìn vĩnh viễn ⇒ pop. Mức 3: trả `len(st)`; đối chiếu với `len(findBuildings(prefix))`.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1, 2, 3
b1 = [Solution().findBuildings(h) for h in [[1], [3, 3, 2, 2, 1], [1, 2, 1, 2], [7, 1, 6, 2, 5, 3, 4]]]
print("Bài 1:", b1); assert b1 == [[0], [1, 3, 4], [3], [0, 2, 4, 6]]
print("Bài 2:"); assert trace_stack([2, 5, 3, 3, 1]) == [1, 3, 4]
def suffix_max_todo(h):
    n = len(h); suf = [0] * (n + 1)
    for i in range(n - 1, -1, -1):
        suf[i] = max(h[i], suf[i + 1])
    return suf
assert suffix_max_todo([4, 2, 3, 1]) == [4, 3, 3, 1, 0]
print("Bài 3:", suffix_max_todo([4, 2, 3, 1]))

Bài 1: [[0], [1, 3, 4], [3], [0, 2, 4, 6]]
Bài 2:
i=0: h=2, pop [] (bị che) -> stack=[0] (cao: [2])
i=1: h=5, pop [0] (bị che) -> stack=[1] (cao: [5])
i=2: h=3, pop [] (bị che) -> stack=[1, 2] (cao: [5, 3])
i=3: h=3, pop [2] (bị che) -> stack=[1, 3] (cao: [5, 3])
i=4: h=1, pop [] (bị che) -> stack=[1, 3, 4] (cao: [5, 3, 1])
Kết quả: [1, 3, 4]
Bài 3: [4, 3, 3, 1, 0]


In [25]:
# Bài 4, 5
def both_sides(heights):
    right = set(Solution().findBuildings(heights))
    left, mx = set(), 0
    for i, h in enumerate(heights):
        if h > mx:
            left.add(i); mx = h
    return sorted(left | right)

class OceanView:
    def __init__(self):
        self.st = []
    def add(self, h):
        while self.st and self.st[-1] <= h:
            self.st.pop()
        self.st.append(h)
        return len(self.st)

rng = random.Random(6)
for _ in range(500):
    h = [rng.randint(1, 6) for _ in range(rng.randint(1, 15))]
    n = len(h)
    brute = [i for i in range(n) if all(h[j] < h[i] for j in range(i + 1, n)) or all(h[j] < h[i] for j in range(i))]
    assert both_sides(h) == brute
    ov = OceanView()
    assert [ov.add(x) for x in h] == [len(ocean_bruteforce(h[:k + 1])) for k in range(n)]
print("Bài 4, 5: OK |", both_sides([1, 3, 2, 4, 2]), [OceanView().add(5)])

Bài 4, 5: OK | [0, 1, 3, 4] [1]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
ans, mx = [], 0
for i in range(len(heights) - 1, -1, -1):
    if heights[i] > mx:
        ans.append(i); mx = heights[i]
ans.reverse()
return ans
```

**Checklist:**
- [ ] So sánh **nghiêm ngặt** `>`?
- [ ] Duyệt từ **phải** (biển ở bên phải)?
- [ ] Đảo kết quả để chỉ số **tăng dần**?
- [ ] Tòa cuối luôn có trong đáp án?
- [ ] Đã test chiều cao bằng nhau, mảng tăng dần, giảm dần, `n = 1`?